# ReLU, fonctions affines par morceaux et splines

Ce notebook accompagne l'exercice 2.3 du cours. Il reprend et réorganise un ancien notebook consacré aux splines. Le point central est le lien exact entre l'espace $\mathcal P_h^1$ et un perceptron utilisant la fonction ReLU.

Nous étudierons successivement la base nodale, sa réalisation par des ReLU, l'interpolation, un problème de moindres carrés et enfin une variante à maillage adaptatif.

## Parcours

1. [L'espace $\mathcal P_h^1$](#espace-p1)
2. [Fonctions chapeau et ReLU](#chapeaux-relu)
3. [Interpolation](#interpolation)
4. [Moindres carrés et apprentissage](#moindres-carres)
5. [Maillage adaptatif](#maillage-adaptatif)
6. [Extensions de l'exercice](#extensions)

In [2]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

print(f"JAX {jax.__version__}, Optax {optax.__version__}")

JAX 0.11.1, Optax 0.2.8


<a id="espace-p1"></a>
## 1. L'espace $\mathcal P_h^1$

Soit un maillage
$$
h:\quad a=t_0<t_1<\cdots<t_n=b.
$$
L'espace $\mathcal P_h^1$ contient les fonctions continues sur $[a,b]$ qui sont affines sur chaque intervalle $[t_{i-1},t_i]$.

1. Vérifier qu'il s'agit d'un espace vectoriel.
2. Écrire les conditions de continuité sur les coefficients des expressions affines locales.
3. Construire les fonctions nodales $\varphi_i$ telles que $\varphi_i(t_j)=\delta_{ij}$.
4. Montrer que toute fonction $v_h\in\mathcal P_h^1$ s'écrit
   $$v_h=\sum_{i=0}^n v_h(t_i)\varphi_i.$$
5. Montrer qu'elle possède aussi une représentation
   $$v_h(t)=\alpha+\beta t+\sum_{i=1}^{n-1}d_i(t-t_i)_+.$$
6. En déduire que les fonctions affines par morceaux permettent d'approcher uniformément toute fonction continue sur $[a,b]$.

<a id="chapeaux-relu"></a>
## 2. Fonctions chapeau et ReLU

La fonction
$$
(t-s)_+=\operatorname{ReLU}(t-s)
$$
est affine de part et d'autre de $s$ et sa pente change en $s$. Pour un nœud intérieur $t_i$, la fonction chapeau s'écrit
$$
\varphi_i(t)=
\frac{(t-t_{i-1})_+}{t_i-t_{i-1}}
-\left(\frac1{t_i-t_{i-1}}+\frac1{t_{i+1}-t_i}\right)(t-t_i)_+
+\frac{(t-t_{i+1})_+}{t_{i+1}-t_i}.
$$

Pour traiter les deux extrémités avec la même formule, nous ajoutons deux nœuds fantômes en prolongeant le premier et le dernier intervalle.

Construisez une fonction `base_chapeaux(maillage, t)` qui renvoie la matrice $(\varphi_i(t_j))_{ij}$. Vérifiez la propriété nodale sur un maillage non uniforme et interprétez la construction comme un MLP à une couche cachée.

In [ ]:
# À compléter.

In [ ]:
# À compléter.

<a id="interpolation"></a>
## 3. Interpolation

Pour une fonction $u$, l'interpolant nodal est
$$
I_hu(t)=\sum_i u(t_i)\varphi_i(t).
$$

Écrivez la machine correspondante et vérifiez qu'elle interpole $u(t)=\sin(\pi t)$ aux nœuds.

In [ ]:
# À compléter.

<a id="moindres-carres"></a>
## 4. Moindres carrés et apprentissage

Nous observons maintenant $u$ en davantage de points que la machine ne possède de coefficients. On minimise
$$
\mathcal L(c)=\frac1{n_{\mathcal D}}\sum_{j=1}^{n_{\mathcal D}}
|u(t_j)-c^\top\varphi(t_j)|^2.
$$

1. Résolvez d'abord le problème par moindres carrés linéaires.
2. Retrouvez ensuite la même solution par apprentissage avec Optax.
3. Comparez cette approximation à l'interpolant précédent. Pourquoi le coût minimal n'est-il généralement pas nul ?

In [6]:
t_observe = jnp.linspace(maillage[0], maillage[-1], 61)
z_observe = u(t_observe)

def cout_coefficients(coefficients, maillage, t, z):
    residu = machine_p1(maillage, coefficients, t) - z
    return jnp.mean(residu**2)

In [ ]:
# À compléter.

In [ ]:
# À compléter.

In [ ]:
# À compléter.

<a id="maillage-adaptatif"></a>
## 5. Maillage adaptatif

Nous fixons les extrémités $a,b$ et écrivons les longueurs des intervalles sous la forme
$$
h_i=(b-a)\,\operatorname{softmax}(q)_i.
$$
Elles sont strictement positives et leur somme vaut $b-a$ : les nœuds restent donc automatiquement ordonnés. Ajustez simultanément les coefficients et le maillage pour approcher $|\sin(\pi t)|$.

In [ ]:
# À compléter.

In [ ]:
# À compléter.

<a id="extensions"></a>
## 6. Extensions de l'exercice

1. Utiliser la formule
   $$
   \sum_{j=0}^k b_jt^j+\sum_i a_i(t-t_i)_+^k
   $$
   pour décrire une machine engendrant des splines de degré $k$. Quelle activation remplace ReLU ?
2. Comment transposer l'idée à $\mathcal P_h^1$ sur un maillage triangulaire ? Pourquoi la géométrie est-elle plus complexe qu'en dimension un ?

## Bilan

L'espace classique $\mathcal P_h^1$ est exactement une classe de machines ReLU dont les ruptures sont fixées par le maillage et dont la dernière couche contient les coefficients. Si le maillage varie, la dépendance par rapport aux paramètres devient non linéaire : on retrouve une machine adaptative.

Cet exemple relie éléments finis, splines et réseaux de neurones sans changer de point de vue mathématique.